# 02. Matrix Notation and Least Squares

The lecture notation writes the model for all observations at once as

$$\mathbf{Y}=\mathbf{X}\boldsymbol\beta+\boldsymbol\epsilon.$$

For a model with an intercept, `Cases`, and `Distance`, each row of $\mathbf X$ is

$$[1,\ Cases_i,\ Distance_i].$$

When $\mathbf X'\mathbf X$ is invertible, the least-squares estimator is

$$\widehat{\boldsymbol\beta}=(\mathbf X'\mathbf X)^{-1}\mathbf X'\mathbf Y.$$

By the end of this notebook, you should be able to:

- build the design matrix for a multiple regression model;
- compute least-squares estimates with the normal-equation formula;
- connect residual variance, $(X'X)^{-1}$, and coefficient standard errors.


The normal equations are

$$\mathbf X'\mathbf X\widehat{\boldsymbol\beta}=\mathbf X'\mathbf Y.$$

The closed-form estimator requires $\mathbf X'\mathbf X$ to be nonsingular. Practically, that means the columns of the design matrix must be linearly independent. If one predictor is an exact linear combination of other predictors, the inverse does not exist and the model is not identifiable in this form.


In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


In [ ]:
delivery = pd.read_csv("data/delivery_time.csv")
y = delivery["Time"].to_numpy()
X = np.column_stack([
    np.ones(len(delivery)),
    delivery["Cases"].to_numpy(),
    delivery["Distance"].to_numpy(),
])

beta_hat = np.linalg.inv(X.T @ X) @ X.T @ y
pd.Series(beta_hat, index=["Intercept", "Cases", "Distance"])


Now compare the matrix calculation with the formula API. They are two interfaces to the same least-squares problem.


In [ ]:
model = smf.ols("Time ~ Cases + Distance", data=delivery).fit()
print(model.params)


In [ ]:
from checks import check_close
print(check_close(beta_hat[1], model.params["Cases"], tol=1e-10, label="matrix Cases coefficient"))
print(check_close(beta_hat[2], model.params["Distance"], tol=1e-10, label="matrix Distance coefficient"))


## Fitted Values, Residuals, and the Hat Matrix

The least-squares fitted values can also be written in matrix form. Starting from

$$
\widehat{\boldsymbol\beta}=(X'X)^{-1}X'Y,
$$

we get

$$
\widehat Y=X\widehat{\boldsymbol\beta}=X(X'X)^{-1}X'Y=HY,
$$

where

$$
H=X(X'X)^{-1}X'
$$

is called the **hat matrix** because it puts the hat on $Y$. The residual vector is

$$
e=Y-\widehat Y=(I-H)Y.
$$

The hat matrix is a projection matrix. Algebraically, use associativity and $(X'X)^{-1}(X'X)=I$ to simplify

$$
H^2=X(X'X)^{-1}X'X(X'X)^{-1}X'.
$$

Once you know $H^2=H$, the same expansion pattern shows how $(I-H)^2$ behaves. This is the matrix algebra behind fitted values and residuals in least squares.

## Residual Variance and Coefficient Variability

With $k$ predictors plus an intercept, the residual degrees of freedom are $n-(k+1)$. If the design matrix has $p=k+1$ columns, the unbiased estimator of $\sigma^2$ is

$$
\widehat{\sigma}^2=MSE=\frac{SSE}{n-p}=\frac{e'e}{n-p}.
$$

The residual standard error is

$$
\widehat{\sigma}=\sqrt{MSE}.
$$

The estimated covariance matrix of $\widehat{\boldsymbol\beta}$ is

$$
MSE(\mathbf X'\mathbf X)^{-1}.
$$


Under the model assumptions,

$$E(\widehat{\boldsymbol\beta})=\boldsymbol\beta, \qquad
\operatorname{Var}(\widehat{\boldsymbol\beta})=\sigma^2(\mathbf X'\mathbf X)^{-1}.$$

If $C=(\mathbf X'\mathbf X)^{-1}$, then the standard error of coefficient $b_j$ is estimated by

$$s\{b_j\}=\sqrt{MSE\,C_{jj}}.$$


In [ ]:
y_hat = X @ beta_hat
resid = y - y_hat
n, p = X.shape
sse = resid @ resid
mse = sse / (n - p)
sigma_hat = np.sqrt(mse)

print(f"SSE = {sse:.4f}")
print(f"MSE = {mse:.4f}")
print(f"residual standard error = {sigma_hat:.4f}")

vcov_beta = mse * np.linalg.inv(X.T @ X)
se_beta = np.sqrt(np.diag(vcov_beta))

summary = pd.DataFrame({
    "estimate": beta_hat,
    "std_error": se_beta,
}, index=["Intercept", "Cases", "Distance"])
summary


The matrix formulas are useful because they show where the standard errors come from. A coefficient can have a large or small standard error depending on residual noise, sample size, and the geometry of the predictors in $\mathbf X$.
